## load and prepare data

In [ ]:
%cd ../..

import sys
import glob
import re

import pandas as pd

sys.path.insert(0, 'evaluation_meldgraph')
from vol_eval_plots import (load_and_prepare_data, select_fcd_control_groups, latest_vol_eval_path,
                            MAIN_3T_CONDITIONS, MAIN_7T_ADAPTED_CONDITIONS)

In [ ]:
# this notebook reports the subject counts before the FCD / control selection, so it applies that
# selection itself, further down
eval_stats_df = load_and_prepare_data(select_fcd_groups=False)

### Supplementary Figure 1

In [ ]:
# publication site names by data source, as load_and_prepare_data() assigns them
site_of_source = dict(eval_stats_df[['source', 'site']].drop_duplicates().values)

clinical_df = pd.read_csv('data/subjects.csv')
clinical_df = clinical_df[clinical_df['source'].isin(site_of_source)]
clinical_df['site'] = clinical_df['source'].map(site_of_source)
clinical_df['site_subj_id'] = clinical_df['site'] + '_' + clinical_df['subject ID'].astype(str)
assert clinical_df['site_subj_id'].is_unique

# one row per subject with include == 1
subjects_df = clinical_df[clinical_df['include'] == 1].set_index('site_subj_id')

In [ ]:
# glob all data/raw/<site>/sub-<subject_id>/ses-7[tT] directories, to check how many have 7T data in the raw data
paths_7traw = glob.glob('data/raw/*/bids/sub-*/ses-7[tT]*', recursive=True)
ids_7traw = [(re.sub(r'.*raw/(.*?)/bids/(sub-.*?)/.*', r'\1', path),
                  re.sub(r'.*raw/(.*?)/bids/(sub-.*?)/.*', r'\2', path)) for path in paths_7traw]
# to dataframe
df_7traw = pd.DataFrame(ids_7traw, columns=['source', 'subj_id'])
df_7traw['site'] = df_7traw['source'].map(site_of_source)
df_7traw['site_subj_id'] = df_7traw['site'] + '_' + df_7traw['subj_id'].astype(str)
df_7traw_ids = set(df_7traw['site_subj_id'])

# drop subjects that don't have 7T raw data
subjects_df = subjects_df[subjects_df.index.isin(df_7traw_ids)]


In [ ]:

stages = {}       # stage name -> set of site_subj_ids
exclusions = []   # (stage not reached, reason, site_subj_id)

stages[0] = set(subjects_df.index) # stage 0: all source pooled subjects

In [ ]:
# stage 1: the healthy controls and the FCD patients with a lesion mask, as 
# select_fcd_control_groups() and vol_eval.py select them
other_pathology_categories = ['category_HS', 
                              'category_mesiotemporal_lesion', 
                              'category_LEAT', 
                              'category_encephalocele', 
                              'category_PNH', 
                              'category_cavernoma', 
                              'category_other']


stage = 1
stages[stage] = set()
for subj in stages[0]:
    group = subjects_df.loc[subj, 'group']
    if group not in ['patient', 'control']:
        reason = f"group '{group}'"
    elif (group == 'patient') and (subjects_df.loc[subj, 'category_FCD'] != 1):
        if any(subjects_df.loc[subj, cat] == 1 for cat in other_pathology_categories):
            reason = 'patient with other pathology'
        elif subjects_df.loc[subj, 'category_nonlesional'] == 1:
            reason = 'patient nonlesional'
        else:
            reason = 'patient without sufficient clinical information'
    else:
        stages[stage].add(subj)
        continue
    exclusions.append((stage, reason, subj))

In [ ]:
# stage 2: MELD-graph input at both 3T and 7T (adapted), from the B0_conditions of each subject in the vol_eval CSV
vol_eval_df = pd.read_csv(latest_vol_eval_path())
vol_eval_df = vol_eval_df[vol_eval_df['model'] == 'meld_graph']
vol_eval_df['site_subj_id'] = vol_eval_df['site'].map(site_of_source) + '_' + vol_eval_df['subject ID'].astype(str)
conditions_of_subject = vol_eval_df.groupby('site_subj_id')['B0_condition'].agg(set)

stage = 2
stages[stage] = set()
for subj in stages[1]:
    conditions = conditions_of_subject.get(subj, set())
    has_3T = bool(conditions & set(MAIN_3T_CONDITIONS))
    has_7T = bool(conditions & set(MAIN_7T_ADAPTED_CONDITIONS))
    if not conditions:
        reason = 'no MELD-graph input at all (not in the vol_eval CSV)'
    elif not has_3T and not has_7T:
        reason = 'no 3T and no 7T MELD-graph input'
    elif not has_3T:
        reason = 'no 3T MELD-graph input'
    elif not has_7T:
        reason = 'no 7T MELD-graph input'
    else:
        stages[stage].add(subj)
        continue
    exclusions.append((stage, reason, subj))

In [ ]:
# stage 3: the exclusion after quality control in load_and_prepare_data()
qc_excluded_ids = {'KCL_sub-RICE075'}
stages[3] = stages[2] - qc_excluded_ids
for subj in stages[2] & qc_excluded_ids:
    exclusions.append((3, 'excluded after quality control', subj))

# the last stage has to be the cohort the other notebooks evaluate, both lists below should be empty
eval_stats_df_fcd = select_fcd_control_groups(eval_stats_df)
print(f"in the final cohort of load_and_prepare_data() only: {sorted(set(eval_stats_df_fcd['site_subj_id']) - stages[3])}")
print(f"in the last stage here only: {sorted(stages[3] - set(eval_stats_df_fcd['site_subj_id']))}")

In [ ]:
# number of subjects at each stage, by group and by site
cohort_flow = pd.DataFrame({
    stage: pd.concat([pd.Series({'n subjects': len(ids)}),
                      subjects_df.loc[sorted(ids), 'group'].value_counts().add_prefix('group: '),
                      subjects_df.loc[sorted(ids), 'site'].value_counts().add_prefix('site: ')])
    for stage, ids in stages.items()}).T.fillna(0).astype(int)
cohort_flow

In [ ]:
# reasons for exclusion at each stage, by group
exclusions_df = pd.DataFrame(exclusions, columns=['stage not reached', 'reason', 'site_subj_id'])
exclusions_df = exclusions_df.join(subjects_df[['site', 'group']], on='site_subj_id')
exclusions_df = exclusions_df.sort_values(['stage not reached', 'reason', 'site_subj_id']).reset_index(drop=True)
pd.crosstab([exclusions_df['stage not reached'], exclusions_df['reason']], exclusions_df['group'], margins=True)

In [ ]:
# every excluded subject
exclusions_df